# Cross-Encoder의 입력 길이와 VAL 순위

ColBERT epoch 5가 각 쿼리에서 뽑은 후보 100개를 Mixed Cross-Encoder epoch 3으로 다시 채점한다. 검색 설정은 토큰당 이웃 1,024개, IVF nprobe 32, ANN 풀 900개다. 같은 체크포인트와 같은 후보 순서에서 **리뷰 다섯 개와 상품 설명을 합친 입력의 상한**만 바꾼다.

기존 SageMaker 커널에서 **Run All**을 실행한다. 첫 코드 셀의 `CROSS_TOKEN_LENGTHS`로 비교할 길이를 고른다. **448**은 학습 기준점, **480**은 중간값, **512**는 이 DistilBERT의 위치 임베딩 상한이다. 짧은 입력에 대한 민감도까지 보려면 384를 추가할 수 있다. 리뷰별 58토큰 상한과 `longest_first`는 학습 설정을 따른다.

In [ ]:
from contextlib import nullcontext
from pathlib import Path
from tempfile import TemporaryDirectory
import hashlib
import json

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

CROSS_TOKEN_LENGTHS = [448, 480, 512]  # 빠른 비교는 [448, 512]. 학습 길이는 자동으로 포함한다.
VAL_LIMIT = 350                        # 빠른 점검은 10. None이면 기존 Mixed VAL 전체.
PROJECT_DIR = Path('/home/ec2-user/SageMaker/20262R0136DATA30400')
MIXED_RUN_ID = None                    # 조건이 같은 실행이 여럿일 때만 출력된 폴더명을 입력한다.
INFER_BATCH_SIZE = None                # None이면 기존 cross_infer_batch. GPU 메모리가 부족하면 16.

RUN_DIR = PROJECT_DIR / 'data304_outputs'
MIXED_ROOT = RUN_DIR / 'cross_hard4_mixed4_lr5e-6'
COLBERT_EPOCH, CROSS_EPOCH = 5, 3
SEARCH = dict(ann_per_token=1024, ann_nprobe=32, ann_pool=900, candidate_topk=100)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16 = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
if DEVICE.type == 'cuda':
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 기존 실험과 VAL 후보

혼합 오답 4+4, 학습률 `5e-6`으로 저장한 실행의 설정과 가중치를 찾는다. 그 실행에서 새로 저장한 VAL 후보와 당시 재사용한 후보를 모두 읽는다. `VAL_LIMIT`을 줄이면 같은 VAL의 앞부분을 사용한다.

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8-sig'))


def signature(value):
    encoded = json.dumps(value, sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(encoded).hexdigest()


def file_sha(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


def find_mixed_run():
    parent = MIXED_ROOT / f'colbert_epoch_{COLBERT_EPOCH:02d}'
    manifests = ([parent / MIXED_RUN_ID / 'settings.json'] if MIXED_RUN_ID is not None
                 else sorted(parent.glob('*/settings.json')))
    matches = []
    for path in manifests:
        if not path.is_file():
            continue
        settings = read_json(path)
        training, mining = settings['training'], settings['mining']
        epoch_dir = path.parent / f'epoch_{CROSS_EPOCH:02d}'
        weights = epoch_dir / 'checkpoints' / 'cross_mixed.pt'
        if (weights.is_file()
                and training['colbert_epoch'] == COLBERT_EPOCH
                and CROSS_EPOCH in training['target_epochs']
                and (training['hard_count'], training['original_count'], training['original_mode']) == (4, 4, 'mixed')
                and training['lr'] == 5e-6
                and (training['config']['cross_tokens'], training['config']['cross_tokens_per_review']) == (448, 58)
                and all(mining[key] == value for key, value in SEARCH.items())
                and training['mining'] == signature(mining)):
            matches.append((path.parent, settings, weights))
    if not matches:
        raise FileNotFoundError(f'Mixed epoch 3의 settings.json과 epoch_03/checkpoints/cross_mixed.pt를 확인하세요. {parent}')
    if len(matches) > 1:
        names = [folder.name for folder, _, _ in matches]
        raise ValueError(f'조건이 같은 실행이 여러 개입니다. 첫 셀의 MIXED_RUN_ID에 폴더명을 지정하세요. {names}')
    return matches[0]


SOURCE_RUN, SAVED, CROSS_WEIGHTS = find_mixed_run()
MINING = SAVED['mining']
TRAIN_CONFIG = SAVED['training']['config']
TRAIN_LENGTH = TRAIN_CONFIG['cross_tokens']
PER_REVIEW = TRAIN_CONFIG['cross_tokens_per_review']
CATALOG_PATH = PROJECT_DIR / 'catalog.json'
GOLD_PATH = PROJECT_DIR / 'train_gold.json'
SPLIT_PATH = RUN_DIR / 'split.json'
for path, expected_hash in ((CATALOG_PATH, MINING['index']['catalog']),
                            (GOLD_PATH, MINING['gold']), (SPLIT_PATH, MINING['split'])):
    if file_sha(path) != expected_hash:
        raise ValueError(f'후보를 만들 때 사용한 파일과 내용이 다릅니다. {path}')

project_config = {}
for config_path in (RUN_DIR / 'aspect_vocabulary_and_config.json', RUN_DIR / 'run_manifest.json'):
    if config_path.is_file():
        project_config.update(read_json(config_path)['config'])
BATCH = project_config['cross_infer_batch'] if INFER_BATCH_SIZE is None else INFER_BATCH_SIZE
if type(BATCH) is not int or BATCH < 1:
    raise ValueError('추론 배치 크기는 양의 정수여야 합니다.')

saved_ids = SAVED['val_query_ids']
split_ids = read_json(SPLIT_PATH)['val_query_ids']
if not saved_ids or saved_ids != split_ids[:len(saved_ids)] or len(set(saved_ids)) != len(saved_ids):
    raise ValueError('저장한 Mixed VAL의 쿼리와 split.json의 순서를 확인하세요.')
if VAL_LIMIT is not None and (type(VAL_LIMIT) is not int or not 1 <= VAL_LIMIT <= len(saved_ids)):
    raise ValueError(f'VAL_LIMIT은 1~{len(saved_ids)} 또는 None으로 지정하세요.')
VAL_IDS = saved_ids[:VAL_LIMIT]
gold_rows = {r['query_id']: r for r in read_json(GOLD_PATH)}
VAL = [gold_rows[qid] for qid in VAL_IDS]
del gold_rows


def metadata_text(item):
    fields = [('title', 'Product'), ('store', 'Store'), ('features', 'Features'), ('description', 'Description')]
    parts = [f'{label}: {item[key].strip()}' for key, label in fields if item[key].strip()]
    return ' [SEP] '.join(parts) or 'Product metadata not available'


catalog = read_json(CATALOG_PATH)
ITEM_IDS = [item['item_id'] for item in catalog]
ITEM_TO_IDX = {iid: i for i, iid in enumerate(ITEM_IDS)}
CAT_TEXTS = [metadata_text(item) for item in catalog]
del catalog
if len(ITEM_IDS) != len(ITEM_TO_IDX) or any(row['item_id'] not in ITEM_TO_IDX for row in VAL):
    raise ValueError('카탈로그의 상품 ID와 VAL 정답을 확인하세요.')

In [ ]:
def load_val_candidates():
    mining_key = signature(MINING)[:20]
    split_key = 'val_' + signature(saved_ids)[:16]  # 일부만 평가해도 원래 350건 캐시를 읽는다.
    roots = []
    for field, fallback in [('candidate_cache', MIXED_ROOT), ('previous_readonly_candidates', RUN_DIR / 'cross_hard_negative')]:
        recorded = Path(SAVED[field])
        if recorded.name != mining_key:
            raise ValueError(f'검색 설정과 후보 캐시의 식별자가 다릅니다. {recorded}')
        # 프로젝트 폴더를 옮겼을 때도 새 경로에서 같은 캐시를 찾는다.
        roots.extend([fallback / 'candidate_cache' / mining_key / split_key, recorded / split_key])
    roots = list(dict.fromkeys(roots))
    shards = {}
    for root in roots:
        for shard in sorted(root.glob('*.npz')):
            shards.setdefault(shard.name, shard)  # 원래 학습 코드도 mixed 캐시를 우선한다.

    n = len(VAL_IDS)
    topk = min(SEARCH['candidate_topk'], len(ITEM_IDS))
    candidates = np.empty((n, topk), dtype=np.int32)
    ranks = np.empty(n, dtype=np.int32)
    filled = np.zeros(n, dtype=bool)
    for path in shards.values():
        start, end = map(int, path.stem.split('_'))
        if not 0 <= start < end <= len(saved_ids):
            raise ValueError(f'VAL 범위가 잘못된 후보 묶음입니다. {path}')
        if start >= n:
            continue
        stop = min(n, end)
        with np.load(path, allow_pickle=False) as shard:
            docs, ann = shard['candidates'], shard['ann_ranks']
            if (not np.array_equal(shard['query_ids'], np.asarray(saved_ids[start:end], dtype=str))
                    or docs.shape != (end - start, topk) or ann.shape != (end - start,)
                    or docs.dtype.kind not in 'iu' or ann.dtype.kind not in 'iu'
                    or np.any((docs < 0) | (docs >= len(ITEM_IDS)))
                    or np.any((ann < 0) | (ann > SEARCH['ann_pool']))):
                raise ValueError(f'쿼리 순서, 후보 또는 ANN 순위가 잘못되었습니다. {path}')
            docs, ann = docs[:stop - start], ann[:stop - start]
            occupied = filled[start:stop]
            if (np.any(candidates[start:stop][occupied] != docs[occupied])
                    or np.any(ranks[start:stop][occupied] != ann[occupied])):
                raise ValueError(f'같은 쿼리에 서로 다른 후보가 저장되어 있습니다. {path}')
            candidates[start:stop], ranks[start:stop] = docs, ann
            filled[start:stop] = True
    if not filled.all():
        i = int(np.flatnonzero(~filled)[0])
        raise FileNotFoundError(f'기존 Mixed VAL 후보에서 {VAL_IDS[i]}가 누락되었습니다. npz 폴더: {roots}')
    if np.any(np.diff(np.sort(candidates, axis=1), axis=1) == 0):
        raise ValueError('한 쿼리의 상위 후보에 중복 상품이 있습니다.')
    return candidates, ranks


CANDIDATES, ANN_RANKS = load_val_candidates()
print(f'Cross 가중치 {CROSS_WEIGHTS}\nVAL {len(VAL)}건 × 후보 {CANDIDATES.shape[1]}개 | {SEARCH}')

## 입력 구성과 잘린 토큰

학습 때와 같이 리뷰별로 토큰을 자른 후 다시 텍스트로 바꾸고, 번호와 `[SEP]`로 연결한다. 그 리뷰 묶음과 후보 상품을 토크나이저에 두 입력으로 전달한다. 전체 길이에는 특수 토큰도 포함된다. 리뷰별 58토큰을 넘겨 앞에서 잘린 부분은 전체 상한을 늘려도 복원되지 않는다.

아래의 **리뷰별 잘림**은 원문 리뷰 단위다. 마지막 표의 **입력 쌍 잘림**은 리뷰별 상한을 먼저 적용한 뒤, 전체 길이 때문에 리뷰 쪽이나 상품 쪽에서 토큰을 더 잘라낸 후보 쌍의 비율이다.

In [ ]:
backbone = TRAIN_CONFIG['backbone']
config = AutoConfig.from_pretrained(backbone)
config.num_labels = 1
TOKENIZER = AutoTokenizer.from_pretrained(backbone, use_fast=True)
if not TOKENIZER.is_fast:
    raise ValueError('리뷰와 상품 쪽의 토큰 수를 집계하려면 학습 때 사용한 fast tokenizer가 필요합니다.')
tokenizer_hash = hashlib.sha256(TOKENIZER.backend_tokenizer.to_str().encode()).hexdigest()
model_limit = min(config.max_position_embeddings, TOKENIZER.model_max_length)
pair_specials = TOKENIZER.num_special_tokens_to_add(pair=True)
requested = [CROSS_TOKEN_LENGTHS] if isinstance(CROSS_TOKEN_LENGTHS, int) else list(CROSS_TOKEN_LENGTHS)
if any(type(x) is not int or not pair_specials <= x <= model_limit for x in [TRAIN_LENGTH, *requested]):
    raise ValueError(f'전체 토큰 수는 {pair_specials}~{model_limit} 범위의 정수여야 합니다.')
LENGTHS = sorted(set([TRAIN_LENGTH, *requested]))


def token_lengths(texts):
    return TOKENIZER(texts, add_special_tokens=False, truncation=False, return_length=True,
                     return_attention_mask=False, return_token_type_ids=False, verbose=False)['length']


def prepare_reviews(row):
    reviews = [r['text'].strip() for r in row['reviews']]
    if len(reviews) != 5:
        raise ValueError(f"리뷰가 다섯 개가 아닌 쿼리입니다. {row['query_id']}")
    original_lengths = token_lengths(reviews)
    chunks = []
    for i, review in enumerate(reviews):
        ids = TOKENIZER.encode(review, add_special_tokens=False, truncation=True, max_length=PER_REVIEW)
        chunks.append(f'Review {i + 1}: ' + TOKENIZER.decode(ids, skip_special_tokens=True))
    joined = ' [SEP] '.join(chunks)
    return joined, token_lengths([joined])[0], sum(n > PER_REVIEW for n in original_lengths)


QUERIES = [prepare_reviews(row) for row in VAL]
doc_lengths = np.zeros(len(ITEM_IDS), dtype=np.int32)
used_docs = np.unique(CANDIDATES)
for start in range(0, len(used_docs), 128):
    docs = used_docs[start:start + 128]
    doc_lengths[docs] = token_lengths([CAT_TEXTS[i] for i in docs])
n_cut = sum(count for _, _, count in QUERIES)
print(f'학습 {TRAIN_LENGTH} | 비교 {LENGTHS} | 모델 상한 {model_limit} | 리뷰별 상한 {PER_REVIEW}')
print(f'리뷰별 잘림 {n_cut:,}/{len(VAL)*5:,} ({n_cut/(len(VAL)*5):.1%})')


class CrossReranker(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = AutoModelForSequenceClassification.from_config(config)


cross = CrossReranker()
cross.load_state_dict(torch.load(CROSS_WEIGHTS, map_location='cpu', weights_only=True, mmap=True), strict=True)
cross = cross.to(DEVICE).eval().requires_grad_(False)

In [ ]:
@torch.inference_mode()
def score_query(i, cross_tokens):
    row = VAL[i]
    query, q_len, n_shortened = QUERIES[i]
    ranked = CANDIDATES[i]  # 기존 MaxSim 순서. Cross 점수 동점 시 이 순서를 보존한다.
    gold = ITEM_TO_IDX[row['item_id']]
    scores, kept_reviews, kept_docs = [], [], []
    for start in range(0, len(ranked), BATCH):
        docs = ranked[start:start + BATCH]
        tokens = TOKENIZER([query] * len(docs), [CAT_TEXTS[j] for j in docs], padding=True,
                           truncation='longest_first', max_length=cross_tokens, return_tensors='pt')
        for j in range(len(docs)):
            side = tokens.sequence_ids(j)
            kept_reviews.append(side.count(0))
            kept_docs.append(side.count(1))
        inputs = {key: value.to(DEVICE) for key, value in tokens.items()}
        with (torch.autocast('cuda', dtype=torch.bfloat16) if USE_BF16 else nullcontext()):
            logits = cross.net(**inputs).logits.reshape(-1)
        scores.extend(logits.float().cpu().tolist())
    if not np.isfinite(scores).all():
        raise ValueError(f"유한하지 않은 Cross 점수입니다. {row['query_id']}, {cross_tokens}")
    final = ranked[np.argsort(-np.asarray(scores), kind='stable')]
    gold_pos = np.flatnonzero(ranked == gold)
    hit = bool(gold_pos.size)
    rank = int(np.flatnonzero(final == gold)[0]) + 1 if hit else None
    removed_q = q_len - np.asarray(kept_reviews)
    removed_d = doc_lengths[ranked] - np.asarray(kept_docs)
    removed = removed_q + removed_d
    full = q_len + doc_lengths[ranked] + pair_specials
    if np.any(removed_q < 0) or np.any(removed_d < 0) or not np.array_equal(removed, np.maximum(full - cross_tokens, 0)):
        raise ValueError(f"토크나이저의 원래 길이와 잘림 집계가 일치하지 않습니다. {row['query_id']}")
    return {
        'query_id': str(row['query_id']), 'gold_item_id': str(row['item_id']),
        'cross_tokens': cross_tokens, 'cross_tokens_per_review': PER_REVIEW,
        'gold_ann_rank_in_pool': int(ANN_RANKS[i]) or None,
        'gold_maxsim_rank_in_topk': int(gold_pos[0]) + 1 if hit else None,
        'cross_gold_rank': rank, 'cross_top1_item_id': str(ITEM_IDS[final[0]]),
        'ann_hit': bool(ANN_RANKS[i]), 'colbert_hit': hit,
        'cross_top1_hit': rank == 1, 'cross_top10_hit': rank is not None and rank <= 10,
        'n_reviews_capped': n_shortened, 'n_pairs': len(ranked),
        'truncated_pairs': int((removed > 0).sum()),
        'review_side_truncated_pairs': int((removed_q > 0).sum()),
        'metadata_side_truncated_pairs': int((removed_d > 0).sum()),
        'total_input_tokens_kept': int((full - removed).sum()),
        'gold_pair_tokens_removed': int(removed[gold_pos[0]]) if hit else None,
    }

## 결과

`stage_summary.csv`에서 **P@1**과 **R@10**은 모든 VAL 쿼리 기준이다. `cross_top1_given_candidate`는 정답이 후보에 들어온 쿼리만 분모로 쓴다. `top1_gained`와 `top1_lost`는 각각 448에서 오답이었다가 정답이 된 쿼리 수와 그 반대다. `pair_trunc_rate`와 `mean_tokens_kept`를 함께 보면 길이를 늘렸을 때 실제로 더 읽은 양을 확인할 수 있다.

길이 하나를 마칠 때마다 쿼리별 결과를 저장하므로 같은 조건으로 재실행하면 완료한 길이부터 재사용한다. 마지막 셀은 요약표와 `query_diagnostics.csv`의 경로를 출력한다.

In [ ]:
def save_csv(frame, path):
    with TemporaryDirectory(dir=path.parent) as tmp:
        staged = Path(tmp) / path.name
        frame.to_csv(staged, index=False)
        staged.replace(path)


weights_hash = file_sha(CROSS_WEIGHTS)
spec = {
    'version': 1, 'source_run': SOURCE_RUN.name, 'cross_sha256': weights_hash,
    'colbert_epoch': COLBERT_EPOCH, 'cross_epoch': CROSS_EPOCH, 'mining': MINING,
    'val_query_ids': VAL_IDS, 'source_val_set': SAVED['val_set'],
    'candidate_sha256': hashlib.sha256(CANDIDATES.tobytes() + ANN_RANKS.tobytes()).hexdigest(),
    'backbone': backbone, 'train_cross_tokens': TRAIN_LENGTH, 'cross_tokens_per_review': PER_REVIEW,
    'cross_infer_batch': BATCH, 'device': DEVICE.type, 'precision': 'bfloat16' if USE_BF16 else 'float32',
    'tokenizer_sha256': tokenizer_hash, 'torch_version': str(torch.__version__),
}
result_root = RUN_DIR / 'cross_token_validation' / SOURCE_RUN.name / signature(spec)[:16]
result_root.mkdir(parents=True, exist_ok=True)
id_types = {key: str for key in ('query_id', 'gold_item_id', 'cross_top1_item_id')}
parts = {}
for length in LENGTHS:
    path = result_root / f'queries_{length}.csv'
    if path.is_file():
        part = pd.read_csv(path, dtype=id_types)
        if part['query_id'].tolist() != list(map(str, VAL_IDS)) or not part['cross_tokens'].eq(length).all():
            raise ValueError(f'저장된 결과의 VAL 순서 또는 입력 길이가 다릅니다. {path}')
        print(f'{length}토큰 결과 재사용', flush=True)
    else:
        part = pd.DataFrame([score_query(i, length) for i in tqdm(range(len(VAL)), desc=f'VAL {length} tokens')])
        save_csv(part, path)
    parts[length] = part.set_index('query_id', verify_integrity=True)

baseline = parts[TRAIN_LENGTH]
base_hits = baseline['cross_top1_hit']
summary_rows = []
for length, frame in parts.items():
    top1 = frame['cross_top1_hit']
    candidate_hits = int(frame['colbert_hit'].sum())
    n_pairs = int(frame['n_pairs'].sum())
    summary_rows.append({
        'cross_tokens': length, 'cross_tokens_per_review': PER_REVIEW, 'n_queries': len(frame),
        'colbert_epoch': COLBERT_EPOCH, 'cross_epoch': CROSS_EPOCH, **SEARCH,
        'ann_recall_at_900': frame['ann_hit'].mean(), 'colbert_hits': candidate_hits,
        'colbert_recall_at_100': frame['colbert_hit'].mean(), 'top1_hits': int(top1.sum()),
        'cross_precision_at_1': top1.mean(), 'cross_recall_at_10': frame['cross_top10_hit'].mean(),
        'cross_top1_given_candidate': top1.sum() / candidate_hits if candidate_hits else np.nan,
        'top1_gained': int((~base_hits & top1).sum()), 'top1_lost': int((base_hits & ~top1).sum()),
        'delta_p_at_1_pp': 100 * (top1.mean() - base_hits.mean()),
        'n_pairs': n_pairs, 'pair_trunc_rate': frame['truncated_pairs'].sum() / n_pairs,
        'review_side_trunc_rate': frame['review_side_truncated_pairs'].sum() / n_pairs,
        'metadata_side_trunc_rate': frame['metadata_side_truncated_pairs'].sum() / n_pairs,
        'mean_tokens_kept': frame['total_input_tokens_kept'].sum() / n_pairs,
    })
prior_key = signature({'cross': weights_hash, 'val': SAVED['val_set'], 'version': 1})[:16]
prior_csv = CROSS_WEIGHTS.parents[1] / 'validation' / prior_key / 'query_diagnostics.csv'
if prior_csv.is_file():
    prior = pd.read_csv(prior_csv, dtype=id_types).set_index('query_id').loc[baseline.index]
    if not prior['gold_maxsim_rank_in_topk'].fillna(0).eq(baseline['gold_maxsim_rank_in_topk'].fillna(0)).all():
        raise ValueError('기존 VAL과 이번 평가에서 ColBERT의 정답 순위가 다릅니다.')
    changed = int((prior['cross_top1_item_id'] != baseline['cross_top1_item_id']).sum())
    print(f"기존 Mixed epoch {CROSS_EPOCH} VAL 대조 | {TRAIN_LENGTH}토큰에서 1위 상품이 달라진 쿼리 {changed}/{len(prior)}")
    if changed:
        print('기존 VAL은 loss 계산용 상품도 마지막 배치에 포함할 수 있습니다. 이번 길이 간 비교에는 재계산한 448 기준값을 씁니다.')

summary = pd.DataFrame(summary_rows)
report = result_root / ('lengths_' + '_'.join(map(str, LENGTHS)))
report.mkdir(exist_ok=True)
save_csv(summary, report / 'stage_summary.csv')
detail = pd.concat([frame.reset_index() for frame in parts.values()], ignore_index=True)
save_csv(detail, report / 'query_diagnostics.csv')
with TemporaryDirectory(dir=report) as tmp:
    staged = Path(tmp) / 'settings.json'
    staged.write_text(json.dumps({**spec, 'cross_weights': str(CROSS_WEIGHTS), 'lengths': LENGTHS},
                                 ensure_ascii=False, indent=2), encoding='utf-8')
    staged.replace(report / staged.name)
display(summary[['cross_tokens', 'top1_hits', 'cross_precision_at_1', 'cross_recall_at_10',
                 'top1_gained', 'top1_lost', 'delta_p_at_1_pp', 'cross_top1_given_candidate']].round(4))
display(summary[['cross_tokens', 'n_pairs', 'pair_trunc_rate', 'review_side_trunc_rate',
                 'metadata_side_trunc_rate', 'mean_tokens_kept']].round(4))
print('요약표', report / 'stage_summary.csv', '\n쿼리별 결과', report / 'query_diagnostics.csv')